In [1]:
# PURPOSE: Download and load the biomarker extraction model.
# WHY: We need to load it once, then reuse it for all 5000 trials.
# The model has two parts:
#   - Base model: Qwen 3.5 (0.8B) — the brain
#   - LoRA adapter: fine-tuned layer for biomarker extraction — the specialization
# Like: a doctor (base) who took extra training in lab tests (LoRA).

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

# Check if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load base model and tokenizer
model_name = "Qwen/Qwen2.5-0.5B"  # base model
adapter_name = "Shubh-0789/biomarker-qwen3.5-0.8b-lora-v2.1"  # LoRA adapter

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(adapter_name, trust_remote_code=True)

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, adapter_name)
model.eval()

print("Model loaded successfully!")

Using device: cpu
Loading tokenizer...


Loading base model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

Loading LoRA adapter...


adapter_config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

Model loaded successfully!


c:\Users\DELL\Desktop\intern_assignment\venv\Lib\site-packages\peft\peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.up

In [11]:
import pandas as pd

df = pd.read_csv("../data/raw/trials_sample.csv")

In [12]:
# PURPOSE: Test the biomarker model on one trial to see what it outputs.
# WHY: Before running 5000 trials, we need to see:
#   - What format does the output come in?
#   - Does it find real biomarkers?
#   - How long does one call take?
# Like: test drive before a road trip.

import time

# Pick a trial that likely has biomarkers
sample = df[df['brief_title'].str.contains('PD-L1|HER2|EGFR|BRCA|biomarker', 
                                            case=False, na=False)].head(3)

for _, row in sample.iterrows():
    # Combine title + primary outcome + secondary outcome
    text_parts = []
    if pd.notna(row['brief_title']):
        text_parts.append(row['brief_title'])
    if pd.notna(row['primary_outcome']):
        text_parts.append(row['primary_outcome'])
    if pd.notna(row['secondary_outcome']):
        text_parts.append(row['secondary_outcome'])
    
    combined_text = " | ".join(text_parts)
    
    # Build prompt
    prompt = f"Extract all biomarkers from the following clinical trial text. List only the biomarker names, separated by commas. If no biomarkers are found, respond with 'None'.\n\nText: {combined_text}\n\nBiomarkers:"
    
    # Run model
    start = time.time()
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            temperature=1.0
        )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    elapsed = time.time() - start
    
    print(f"Title: {row['brief_title'][:80]}...")
    print(f"Output: {response.strip()}")
    print(f"Time: {elapsed:.2f}s")
    print()

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


IndexError: index out of range in self

In [13]:
# PURPOSE: Find the correct base model from the adapter config.
# WHY: LoRA adapters must be loaded on top of the EXACT base model
# they were trained on. Wrong base = crash.
# Like: putting a phone case designed for iPhone on a Samsung.

from peft import PeftConfig

config = PeftConfig.from_pretrained("Shubh-0789/biomarker-qwen3.5-0.8b-lora-v2.1")
print(f"Base model name: {config.base_model_name_or_path}")
print(f"Task type: {config.task_type}")
print(f"\nFull config:")
print(config)

Base model name: Qwen/Qwen3.5-0.8B
Task type: CAUSAL_LM

Full config:
LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping={'base_model_class': 'Qwen3_5ForConditionalGeneration', 'parent_library': 'transformers.models.qwen3_5.modeling_qwen3_5', 'unsloth_fixed': True}, peft_version='0.18.1', base_model_name_or_path='Qwen/Qwen3.5-0.8B', revision=None, inference_mode=True, r=8, target_modules={'up_proj', 'q_proj', 'o_proj', 'k_proj', 'v_proj', 'down_proj', 'gate_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None

In [14]:
# PURPOSE: Reload with the correct base model Qwen/Qwen3.5-0.8B.
# WHY: The LoRA adapter was trained on this exact model.
# Wrong base model = index errors and garbage output.

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Free old model from memory
del model, base_model
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

base_model_name = "Qwen/Qwen3.5-0.8B"
adapter_name = "Shubh-0789/biomarker-qwen3.5-0.8b-lora-v2.1"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, adapter_name)
model.eval()

print("Model loaded successfully!")

Using device: cpu
Loading tokenizer...


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

...

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/1.75G [00:00<?, ?B/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loading LoRA adapter...
Model loaded successfully!


c:\Users\DELL\Desktop\intern_assignment\venv\Lib\site-packages\peft\peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.down_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.down_proj.lora_B.default.weight', 'base_model.model.model.layers.1.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.1.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.1.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.1.mlp.up_proj.lora_B.default.weight', 'base_model.model.model.layers.1.mlp.down_proj.lora_A.default.weight', 'base_model.model.model.layers.1.mlp.down_proj.lora_B.default.weight'

In [15]:
# PURPOSE: Check library versions and try loading with unsloth compatibility.
# WHY: The adapter config says 'unsloth_fixed': True, which means it was
# saved with the Unsloth library. We might need to install it.

import peft
import transformers
print(f"peft version: {peft.__version__}")
print(f"transformers version: {transformers.__version__}")
print(f"torch version: {torch.__version__}")

# Check how many adapter weights actually loaded
adapter_params = [n for n, p in model.named_parameters() if 'lora' in n.lower()]
print(f"\nLoRA parameters found in model: {len(adapter_params)}")
if adapter_params:
    print("Sample params:")
    for p in adapter_params[:5]:
        print(f"  {p}")

peft version: 0.19.1
transformers version: 5.9.0
torch version: 2.12.0+cpu

LoRA parameters found in model: 192
Sample params:
  base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight
  base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight
  base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight
  base_model.model.model.layers.0.mlp.up_proj.lora_B.default.weight
  base_model.model.model.layers.0.mlp.down_proj.lora_A.default.weight


In [16]:
# PURPOSE: Test the model on a few trials to see what output format it gives.
# WHY: We need to see the output before building the batch pipeline.

import time

# Find trials likely to have biomarkers
sample = df[df['brief_title'].str.contains('PD-L1|HER2|EGFR|BRCA|biomarker', 
                                            case=False, na=False)].head(3)

# If no matches, just use first 3 rows
if len(sample) == 0:
    sample = df.head(3)

for _, row in sample.iterrows():
    # Combine text fields
    text_parts = []
    if pd.notna(row['brief_title']):
        text_parts.append(row['brief_title'])
    if pd.notna(row['primary_outcome']):
        text_parts.append(row['primary_outcome'])
    
    combined = " ".join(text_parts)[:500]  # limit length
    
    prompt = f"Extract all biomarkers mentioned in the following clinical trial text. List only biomarker names separated by commas. If none found, say 'None'.\n\nText: {combined}\n\nBiomarkers:"
    
    start = time.time()
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    # Move to same device as model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
        )
    
    # Decode only the NEW tokens (skip the prompt)
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    elapsed = time.time() - start
    
    print(f"Title: {row['brief_title'][:80]}...")
    print(f"Model output: '{response.strip()}'")
    print(f"Time: {elapsed:.1f}s")
    print()

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Title: Study of Efficacy and Safety in Premenopausal Women With Hormone Receptor Positi...
Model output: '<think>
Thinking Process:

1.  **Analyze the Request:**
    *   Task: Extract all biomarkers mentioned in the provided clinical trial text.
    *   Output Format: List only biomarker names separated by commas.
    *   Constraint: If none found, say 'None'.
    *   Input Text: "Study of Efficacy and Safety in Premenopausal Women With Hormone Receptor Positive, HER2-negative Advanced Breast Cancer Progress Free'
Time: 437.6s



[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Title: Curcumin Biomarkers...
Model output: 'Curcumin, Ribonucleic acid (RNA) level, Apoptosis

<think>

</think>

Curcumin, Ribonucleic acid (RNA) level, Apoptosis'
Time: 120.8s

Title: A Study of Tucatinib Given Before Surgery to People With HER2+ Cancers That Have...
Model output: '<think>
Thinking Process:

1.  **Analyze the Request:**
    *   Task: Extract all biomarkers mentioned in the provided clinical trial text.
    *   Output Format: List only biomarker names separated by commas.
    *   Constraint: If none found, say 'None'.
    *   Input Text: "A Study of Tucatinib Given Before Surgery to People With HER2+ Cancers That Have Spread to the Brain Maximum observed plasma concentration ('
Time: 620.5s



In [18]:
# PURPOSE: Fix two issues:
#   1) Disable thinking mode (the <think> tags waste tokens and time)
#   2) Reduce max tokens and add better prompt format
# WHY: 437 seconds per trial is unusable. We need faster output.

import time, re

# Test with thinking disabled
sample_row = df[df['brief_title'].str.contains('HER2', case=False, na=False)].iloc[0]

text_parts = []
if pd.notna(sample_row['brief_title']):
    text_parts.append(sample_row['brief_title'])
if pd.notna(sample_row['primary_outcome']):
    text_parts.append(sample_row['primary_outcome'])
combined = " ".join(text_parts)[:300]

# Use /no_think to disable reasoning mode
prompt = f"/no_think Extract biomarkers from this clinical trial text. Return ONLY a comma-separated list of biomarker names. If none, return 'None'.\n\nText: {combined}\n\nBiomarkers:"

start = time.time()
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,  # reduced — biomarker list should be short
        do_sample=False,
    )

new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True)

# Strip any <think> tags if they still appear
response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()

elapsed = time.time() - start
print(f"Title: {sample_row['brief_title'][:80]}...")
print(f"Output: '{response}'")
print(f"Time: {elapsed:.1f}s")

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Title: Study of Efficacy and Safety in Premenopausal Women With Hormone Receptor Positi...
Output: 'None'
Time: 48.7s


In [19]:
# PURPOSE: Test on multiple trials to see if model finds biomarkers correctly.
# WHY: One "None" result isn't enough — we need to see it find real biomarkers too.

import time

# Pick diverse trials
test_titles = ['HER2', 'PD-L1', 'EGFR', 'BRCA', 'PSA']
results = []

for keyword in test_titles:
    match = df[df['brief_title'].str.contains(keyword, case=False, na=False)]
    if len(match) > 0:
        row = match.iloc[0]
        
        text_parts = []
        if pd.notna(row['brief_title']):
            text_parts.append(row['brief_title'])
        if pd.notna(row['primary_outcome']):
            text_parts.append(row['primary_outcome'])
        combined = " ".join(text_parts)[:400]
        
        prompt = f"/no_think Extract biomarkers from this clinical trial text. Return ONLY a comma-separated list of biomarker names. If none, return 'None'.\n\nText: {combined}\n\nBiomarkers:"
        
        start = time.time()
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=50,
                do_sample=False,
            )
        
        new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True)
        response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()
        elapsed = time.time() - start
        
        print(f"[{keyword}] Title: {row['brief_title'][:70]}...")
        print(f"  Output: '{response}'")
        print(f"  Time: {elapsed:.1f}s")
        print()

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


[HER2] Title: Study of Efficacy and Safety in Premenopausal Women With Hormone Recep...
  Output: 'None'
  Time: 206.9s



[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


[PD-L1] Title: QUILT-3.044: NANT Non-small Cell Lung Cancer (NSCLC) Vaccine: Combinat...
  Output: '<think>
Thinking Process:

1.  **Analyze the Request:**
    *   Task: Extract biomarkers from the provided clinical trial text.
    *   Output Format: A comma-separated list of biomarker names.'
  Time: 879.4s



[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


[EGFR] Title: Gemcitabine/Cisplatin +/-Cetuximab in Patients With Locally Advanced o...
  Output: 'None'
  Time: 71.3s



[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


[BRCA] Title: Genetic Counseling in Women at Risk for BRCA1 or BRCA2 Mutations...
  Output: 'None'
  Time: 66.1s

[PSA] Title: The Prevalence of Local Immunoglobulin E (IgE) Elevation and Its Effec...
  Output: '</think>

None'
  Time: 58.0s

